In [17]:
# %pip install google-play-scraper
# %pip install pandas
# %pip install tensorflow

In [18]:
from google_play_scraper import Sort, reviews, reviews_all, app
import json, pprint, time, re, nltk
import pandas as pd
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

In [19]:
app_args = {
    "app_id": "com.btpn.dc",
    "lang": "en",
    "country": "id"
}

In [20]:
app_info = app(**app_args)

pd.DataFrame([app_info])
# print(json.dumps(app_info, indent=2))

,title,description,descriptionHTML,summary,installs,minInstalls,realInstalls,score,ratings,reviews,...,contentRatingDescription,adSupported,containsAds,released,lastUpdatedOn,updated,version,comments,appId,url
0,Jenius,The simple way to manage your life/finance wit...,The simple way to manage your life/finance wit...,The simple way to manage your life/finance wit...,"10,000,000+",10000000,13761625,3.27501,203279,129301,...,None,False,False,"Aug 10, 2016","Mar 13, 2026",1773408818,4.64.0,[],com.btpn.dc,https://play.google.com/store/apps/details?id=...


In [21]:
# pretty print 5 reviews based on 1* rating
result, continuation_token = reviews(
    **app_args,
    sort=Sort.RATING, # defaults to Sort.NEWEST
    count=5, # defaults to 100
    filter_score_with=1 # defaults to None(means all score)
)

result, _ = reviews(
    app_args['app_id'],
    continuation_token=continuation_token # defaults to None(load from the beginning)
)

pprint.pprint(result)

[{'appVersion': '4.63.1',
  'at': datetime.datetime(2026, 2, 23, 23, 5, 21),
  'content': 'It was good. but getting slower now. even scanning QRIS took '
             'longer compared to other mobile banking apps.',
  'repliedAt': datetime.datetime(2026, 2, 23, 23, 7, 19),
  'replyContent': 'Hi Benny Herlambang. We apologize for the inconvenience. '
                  'The information you provided has been noted and will be '
                  'used as feedback for Jenius to provide/improve better '
                  'services and facilities.',
  'reviewCreatedVersion': '4.63.1',
  'reviewId': '8a566d92-07ef-4275-9731-49bc9c34a2f3',
  'score': 1,
  'thumbsUpCount': 0,
  'userImage': 'https://play-lh.googleusercontent.com/a-/ALV-UjX8vrwAXJftZ8y96Nu6-Q1tOmHECQ9TuedRZLRByohZF_bVgpe0Ew',
  'userName': 'Benny Herlambang'},
 {'appVersion': '4.55.0',
  'at': datetime.datetime(2025, 12, 11, 20, 12, 22),
  'content': 'Nothing works, when i ask for help no one knows what to do, '
             'no

In [22]:
# collect reviews with loop safeguards
all_reviews = []
review_ids = set()

target_count = 10000
batch_size = 200
max_batches_per_sort = (target_count // batch_size) * 3
max_runtime_seconds = 20 * 60
window_size = 10
min_new_per_window = 20

start_time = time.time()

for sort_type in [Sort.NEWEST, Sort.MOST_RELEVANT]:
  continuation_token = None
  seen_tokens = set()
  new_in_window = 0
  consecutive_errors = 0

  for batch_idx in range(1, max_batches_per_sort + 1):
    if len(all_reviews) >= target_count:
      break

    if time.time() - start_time > max_runtime_seconds:
      print("Runtime limit reached. Stopping scrape early.")
      break

    try:
      result, next_token = reviews(
        **app_args,
        sort=sort_type,
        count=batch_size,
        continuation_token=continuation_token
      )
      consecutive_errors = 0
    except Exception as err:
      consecutive_errors += 1
      print(f"{sort_type.name} batch {batch_idx} failed: {err}")
      if consecutive_errors >= 3:
        print(f"{sort_type.name}: too many consecutive errors, stopping.")
        break
      time.sleep(2)
      continue

    if len(result) == 0:
      print(f"{sort_type.name}: no results returned, stopping.")
      break

    prev_count = len(all_reviews)

    for review in result:
      if len(all_reviews) >= target_count:
        break

      review_id = review.get('reviewId')
      if review_id and review_id not in review_ids:
        review_ids.add(review_id)
        all_reviews.append(review)

    added = len(all_reviews) - prev_count
    new_in_window += added
    print(f"{sort_type.name} batch {batch_idx}: +{added} new, total={len(all_reviews)}")

    if len(all_reviews) >= target_count:
      break

    if next_token is None:
      print(f"{sort_type.name}: no continuation token, source exhausted.")
      break

    token_key = repr(next_token)
    if token_key in seen_tokens:
      print(f"{sort_type.name}: repeated continuation token detected, stopping to avoid loop.")
      break

    seen_tokens.add(token_key)
    continuation_token = next_token

    if batch_idx % window_size == 0:
      if new_in_window < min_new_per_window:
        print(f"{sort_type.name}: only {new_in_window} new reviews in last {window_size} batches, stopping.")
        break
      new_in_window = 0

    time.sleep(1.2)

  if len(all_reviews) >= target_count:
    break

  if time.time() - start_time > max_runtime_seconds:
    break

df = pd.DataFrame(all_reviews[:target_count])
print(f"Total reviews collected: {len(df)}.")
if len(df) < target_count:
  print("Target not fully reached. Try running again later for newer reviews.")

NEWEST batch 1: +200 new, total=200
NEWEST batch 2: +200 new, total=400
NEWEST batch 3: +200 new, total=600
NEWEST batch 4: +200 new, total=800
NEWEST batch 5: +200 new, total=1000
NEWEST batch 6: +200 new, total=1200
NEWEST batch 7: +200 new, total=1400
NEWEST batch 8: +200 new, total=1600
NEWEST batch 9: +200 new, total=1800
NEWEST batch 10: +200 new, total=2000
NEWEST batch 11: +200 new, total=2200
NEWEST batch 12: +200 new, total=2400
NEWEST batch 13: +200 new, total=2600
NEWEST batch 14: +200 new, total=2800
NEWEST batch 15: +200 new, total=3000
NEWEST batch 16: +200 new, total=3200
NEWEST batch 17: +200 new, total=3400
NEWEST batch 18: +200 new, total=3600
NEWEST batch 19: +200 new, total=3800
NEWEST batch 20: +200 new, total=4000
NEWEST batch 21: +200 new, total=4200
NEWEST batch 22: +200 new, total=4400
NEWEST batch 23: +200 new, total=4600
NEWEST batch 24: +200 new, total=4800
NEWEST batch 25: +200 new, total=5000
NEWEST batch 26: +200 new, total=5200
NEWEST batch 27: +200 new

In [23]:
# create label sentiment from ratings
def label_sentiment(score):
  if score >= 4:
    return 'positive'
  elif score == 3:
    return 'neutral'
  else:
    return 'negative'

df['sentiment'] = df['score'].apply(label_sentiment)

print(df['sentiment'].value_counts(normalize=True))

sentiment
negative    0.5377
positive    0.4114
neutral     0.0509
Name: proportion, dtype: float64


In [24]:
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Istamosh\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Istamosh\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\Istamosh\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [25]:
def preprocess_text(text):
  text = text.lower()

  text = re.sub(r'http\S+|www\S+|https\S+', '', text)
  text = re.sub(r'@\w+', '', text)
  text = re.sub(r'[^a-zA-Z\s]', '', text)

  text = ' '.join(text.split())

  stop_words = set(stopwords.words('english'))
  lemmatizer = WordNetLemmatizer()

  words = text.split()
  words = [lemmatizer.lemmatize(word) for word in words if word not in stop_words]

  return ' '.join(words)

df['cleaned_content'] = df['content'].apply(preprocess_text)

In [26]:
import os
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier

# Sequence of available options
algorithm_pool = {
    'Logistic Regression': lambda: LogisticRegression(
        max_iter=2000, C=2.0, class_weight='balanced', random_state=42
    ),
    'Naive Bayes': lambda: MultinomialNB(alpha=0.5),
    'SVM': lambda: LinearSVC(C=1.5, class_weight='balanced', random_state=42),
    'Random Forest': lambda: RandomForestClassifier(
        n_estimators=300, min_samples_split=4, class_weight='balanced_subsample',
        random_state=42, n_jobs=-1
    )
}

feature_pool = {
    'TF-IDF': lambda: TfidfVectorizer(
        max_features=30000, ngram_range=(1, 1), min_df=3, max_df=0.9, sublinear_tf=True
    ),
    'BoW': lambda: CountVectorizer(
        max_features=30000, ngram_range=(1, 1), min_df=3, max_df=0.9
    ),
    'TF-IDF+NG': lambda: TfidfVectorizer(
        max_features=40000, ngram_range=(1, 2), min_df=2, max_df=0.95, sublinear_tf=True
    )
}

label_options = ['3-class', '5-class']
split_options = ['80/20', '70/30', 'kfold-5']

print('Available algorithm options:', ', '.join(algorithm_pool.keys()))
print('Available feature options  :', ', '.join(feature_pool.keys()))
print('Available label options    :', ', '.join(label_options))
print('Available split options    :', ', '.join(split_options))

def build_target(df_input, label_mode):
    if label_mode == '3-class':
        return df_input['sentiment'].astype(str)
    if label_mode == '5-class':
        return df_input['score'].astype(int).astype(str)
    raise ValueError('label_mode must be either 3-class or 5-class')

# Keep only rows with non-empty cleaned text
model_df = df.copy()
model_df['cleaned_content'] = model_df['cleaned_content'].fillna('').astype(str)
model_df = model_df[model_df['cleaned_content'].str.len() > 0].reset_index(drop=True)
print(f'Modeling rows available: {len(model_df):,}')

Available algorithm options: Logistic Regression, Naive Bayes, SVM, Random Forest
Available feature options  : TF-IDF, BoW, TF-IDF+NG
Available label options    : 3-class, 5-class
Available split options    : 80/20, 70/30, kfold-5
Modeling rows available: 9,991


In [27]:
# 3 distinct experiments with different combinations
experiments = [
    {
        'Experiment': 1,
        'Algorithm': 'Naive Bayes',
        'Features': 'TF-IDF',
        'Labeling': '3-class',
        'Split': '80/20'
    },
    {
        'Experiment': 2,
        'Algorithm': 'Logistic Regression',
        'Features': 'BoW',
        'Labeling': '3-class',
        'Split': '70/30'
    },
    {
        'Experiment': 3,
        'Algorithm': 'SVM',
        'Features': 'TF-IDF+NG',
        'Labeling': '3-class',
        'Split': 'kfold-5'
    }
]

os.makedirs('data/experiments', exist_ok=True)

results = []
reports = {}
exported_files = []

base_cols = ['content', 'cleaned_content', 'score', 'sentiment']
all_indices = np.arange(len(model_df))

for exp in experiments:
    exp_no = exp['Experiment']
    algo_name = exp['Algorithm']
    feat_name = exp['Features']
    label_mode = exp['Labeling']
    split_mode = exp['Split']

    X_text = model_df['cleaned_content']
    y = build_target(model_df, label_mode)

    pipeline = Pipeline([
        ('vectorizer', feature_pool[feat_name]()),
        ('model', algorithm_pool[algo_name]())
    ])

    if split_mode in ['80/20', '70/30']:
        test_size = 0.2 if split_mode == '80/20' else 0.3

        X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
            X_text, y, all_indices,
            test_size=test_size,
            random_state=42,
            stratify=y
        )

        pipeline.fit(X_train, y_train)
        y_pred = pipeline.predict(X_test)
        y_train_pred = pipeline.predict(X_train)

        train_acc = accuracy_score(y_train, y_train_pred)
        test_acc = accuracy_score(y_test, y_pred)
        macro_f1 = f1_score(y_test, y_pred, average='macro')

        results.append({
            'Experiment': exp_no,
            'Algorithm': algo_name,
            'Features': feat_name,
            'Labeling': label_mode,
            'Split': split_mode,
            'Accuracy': test_acc,
            'F1-Score': macro_f1,
            'Train Accuracy': train_acc
        })

        reports[exp_no] = classification_report(y_test, y_pred, digits=4)

        train_rows = model_df.iloc[idx_train][base_cols].copy()
        train_rows['label_used'] = y_train.values
        test_rows = model_df.iloc[idx_test][base_cols].copy()
        test_rows['label_used'] = y_test.values

        pred_rows = test_rows.copy()
        pred_rows['predicted_label'] = y_pred

        train_path = f'data/experiments/exp{exp_no}_train.csv'
        test_path = f'data/experiments/exp{exp_no}_test.csv'
        pred_path = f'data/experiments/exp{exp_no}_predictions.csv'

        train_rows.to_csv(train_path, index=False)
        test_rows.to_csv(test_path, index=False)
        pred_rows.to_csv(pred_path, index=False)

        exported_files.extend([train_path, test_path, pred_path])

    elif split_mode == 'kfold-5':
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        y_pred_cv = cross_val_predict(pipeline, X_text, y, cv=skf, n_jobs=-1)

        cv_acc = accuracy_score(y, y_pred_cv)
        cv_f1 = f1_score(y, y_pred_cv, average='macro')

        results.append({
            'Experiment': exp_no,
            'Algorithm': algo_name,
            'Features': feat_name,
            'Labeling': label_mode,
            'Split': split_mode,
            'Accuracy': cv_acc,
            'F1-Score': cv_f1,
            'Train Accuracy': np.nan
        })

        reports[exp_no] = classification_report(y, y_pred_cv, digits=4)

        cv_rows = model_df[base_cols].copy()
        cv_rows['label_used'] = y.values
        cv_rows['predicted_label'] = y_pred_cv
        cv_path = f'data/experiments/exp{exp_no}_kfold_predictions.csv'
        cv_rows.to_csv(cv_path, index=False)
        exported_files.append(cv_path)

    else:
        raise ValueError(f'Unsupported split mode: {split_mode}')

results_df = pd.DataFrame(results).sort_values('Experiment').reset_index(drop=True)

In [28]:
# Render result table similar to requested format
display_df = results_df.copy()
display_df['Accuracy'] = (display_df['Accuracy'] * 100).map(lambda x: f'{x:.2f}%')
display_df['F1-Score'] = display_df['F1-Score'].map(lambda x: f'{x:.4f}')
display_df['Train Accuracy'] = display_df['Train Accuracy'].apply(
    lambda x: 'N/A (k-fold)' if pd.isna(x) else f'{x * 100:.2f}%'
 )

result_columns = [
    'Experiment', 'Algorithm', 'Features', 'Labeling',
    'Split', 'Accuracy', 'F1-Score', 'Train Accuracy'
 ]
display_df = display_df[result_columns]

# Build a markdown-style text table without extra dependencies
headers = display_df.columns.tolist()
table_lines = []
table_lines.append(' | '.join(headers))
table_lines.append(' | '.join(['---'] * len(headers)))
for _, row in display_df.iterrows():
    table_lines.append(' | '.join(str(row[h]) for h in headers))

print('\n'.join(table_lines))

print('\nDetailed classification report per experiment:')
for exp_no in sorted(reports.keys()):
    print(f'\nExperiment {exp_no}')
    print(reports[exp_no])

print('\nExported reproducibility files:')
for path in exported_files:
    print(f'- {path}')

trainable_rows = results_df['Train Accuracy'].dropna()
if len(trainable_rows) > 0:
    best_train_acc = trainable_rows.max()
    print(f"\nBest holdout training accuracy: {best_train_acc:.2%}")
    if best_train_acc < 0.92:
        print('Note: no holdout experiment reached 92% training accuracy yet. Try stronger n-grams or hyperparameter tuning.')

Experiment | Algorithm | Features | Labeling | Split | Accuracy | F1-Score | Train Accuracy
--- | --- | --- | --- | --- | --- | --- | ---
1 | Naive Bayes | TF-IDF | 3-class | 80/20 | 88.14% | 0.6044 | 89.41%
2 | Logistic Regression | BoW | 3-class | 70/30 | 82.15% | 0.6577 | 95.72%
3 | SVM | TF-IDF+NG | 3-class | kfold-5 | 87.33% | 0.6367 | N/A (k-fold)

Detailed classification report per experiment:

Experiment 1
              precision    recall  f1-score   support

    negative     0.8456    0.9628    0.9004      1075
     neutral     0.0000    0.0000    0.0000       102
    positive     0.9429    0.8844    0.9127       822

    accuracy                         0.8814      1999
   macro avg     0.5962    0.6157    0.6044      1999
weighted avg     0.8425    0.8814    0.8595      1999


Experiment 2
              precision    recall  f1-score   support

    negative     0.8820    0.8202    0.8500      1613
     neutral     0.1646    0.3399    0.2217       153
    positive     0.9205 